In [0]:
# Configuración general del laboratorio
 
from datetime import datetime, UTC
from pyspark.sql.functions import col
import re
from pyspark.sql import functions as F
from pyspark.sql.window import Window
 
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

## Descripción
Genera un conteo de registros por zona geográfica en la capa Silver.

## Objetivo
Verificar integridad y distribución de datos por zona.

## Entrada
- Tabla: `SILVER_TABLE`

## Procesamiento
- Agrupación por `zona`
- Conteo de registros
- Ordenamiento por `zona`

## Salida
- Conteo por zona (visualización)

## Capa
**Silver (Validación)**


# Selección y Limpieza Final (`Clean_Gold_Data`)

## Descripción
Filtra y selecciona variables relevantes, eliminando registros con valores nulos generados por las ventanas deslizantes.

## Objetivo
Garantizar un dataset completo y consistente para el entrenamiento del modelo ML.

## Entrada
- DataFrame: `df_gold`

## Procesamiento
- Selección de variables clave (features + target)
- Eliminación de nulos en variables críticas generadas por lags y ventanas

## Salida
- DataFrame: `df_gold_selected`

## Capa
**Gold (Preparación ML)**


In [0]:
df_gold_selected = (
    df_gold
    .select(
        "zona",
        "pais",
        "tipo_producto",
        "canal_venta",
        "prioridad",
        "fecha_pedido",
        "id_pedido",
        "ANIO",
        "MES",
        "DIA",
        "Unidades",
        "precio_unitario",
        "importe_venta_total",
        "importe_coste_total",
        "lag_venta_1",
        "lag_venta_2",
        "lag_venta_3",
        "var_venta_1m",
        "var_venta_3m",
        "ma_3",
        "ma_7",
        "stddev_ventas_7m",
        "var_unidades_1m",
        "var_coste_1m",
        "venta_vs_ma_7",
        "margen_ganancia",
        "dias_entrega",
        "target_alta_venta"
    )
    .filter(F.col("lag_venta_3").isNotNull())
    .filter(F.col("var_venta_1m").isNotNull())
    .filter(F.col("var_venta_3m").isNotNull())
    .filter(F.col("ma_7").isNotNull())
    .filter(F.col("stddev_ventas_7m").isNotNull())
    .filter(F.col("margen_ganancia").isNotNull())
    .filter(F.col("dias_entrega").isNotNull())
)


# Persistencia Capa Gold (`Persist_Gold_Layer`)

## Descripción
Guarda el dataset final (features + target) en Delta Lake como tabla Gold.

## Objetivo
Disponibilizar datos listos para analítica avanzada y modelos ML.

## Entrada
- DataFrame: `df_gold_selected`

## Procesamiento
- Drop tabla si existe
- Escritura en formato Delta (`overwrite`)

## Salida
- Tabla: `GOLD_TABLE`

## Capa
**Gold**


In [0]:
spark.sql(f"DROP TABLE IF EXISTS {GOLD_TABLE}")
 
(
    df_gold_selected.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_TABLE)
)
 
print(f"Tabla Gold creada: {GOLD_TABLE}")

# Visualización Capa Gold (`View_Gold_Data`)

## Descripción
Visualiza la evolución temporal de las métricas analíticas generadas (importe de venta, medias móviles, variaciones) desde la capa Gold.

## Objetivo
Analizar tendencias comerciales y validar el comportamiento de las variables generadas para ML.

## Entrada
- Tabla: `GOLD_TABLE`

## Procesamiento
- Lectura de datos desde Gold
- Ordenamiento por `zona`, `fecha_pedido`
- Visualización gráfica en Databricks

## Salida
- Vista tabular / gráfico de series temporales de ventas

## Capa
**Gold (Análisis / Validación)**


In [0]:
display(spark.table(GOLD_TABLE).orderBy("zona", "fecha_pedido").limit(5))

# Análisis de Rentabilidad por Zona (`Analyze_Profitability`)

## Descripción
Calcula el importe de venta promedio, el margen de ganancia promedio y la variabilidad de ventas por zona geográfica y tipo de producto.

## Objetivo
Identificar las zonas y categorías de producto con mayor rentabilidad y menor volatilidad comercial.

## Entrada
- Tabla: `GOLD_TABLE`

## Procesamiento
- Agrupación por `zona` y `tipo_producto`
- Cálculo de:
  - `avg_venta`: importe de venta promedio
  - `avg_margen`: margen de ganancia promedio
  - `avg_volatilidad_ventas`: variabilidad promedio de las ventas
- Ordenamiento por importe de venta promedio descendente

## Salida
- Tabla comparativa de rentabilidad por zona y producto

## Capa
**Gold (Análisis)**


In [0]:
display(
    spark.sql(f"""
    SELECT 
        zona,
        tipo_producto,
        ROUND(AVG(importe_venta_total), 2)  AS avg_venta,
        ROUND(AVG(margen_ganancia), 4)       AS avg_margen,
        ROUND(AVG(stddev_ventas_7m), 6)      AS avg_volatilidad_ventas
    FROM {GOLD_TABLE}
    GROUP BY zona, tipo_producto
    ORDER BY avg_venta DESC
    limit(5)
    """)
)


# Análisis de Tendencia de Ventas por Zona (`Analyze_Sales_Trend`)

## Descripción
Consulta la evolución temporal del importe de ventas de una zona seleccionada, junto con métricas derivadas.

## Objetivo
Analizar la tendencia, el comportamiento mensual y la relación con la media móvil para una zona específica.

## Entrada
- Tabla: `GOLD_TABLE`

## Procesamiento
- Filtro por `zona = 'Europe'` (ajustable según análisis requerido)
- Selección de variables: `importe_venta_total`, `ma_7`, `var_venta_1m`, `margen_ganancia`
- Ordenamiento por `fecha_pedido`

## Salida
- Serie temporal de ventas para la zona seleccionada

## Capa
**Gold (Análisis)**


In [0]:
display(
    spark.sql(f"""
    SELECT 
        zona,
        fecha_pedido,
        tipo_producto,
        importe_venta_total,
        ma_7,
        var_venta_1m,
        margen_ganancia
    FROM {GOLD_TABLE}
    WHERE zona = 'Europa'
    ORDER BY fecha_pedido
    limit(5)
    """))


# Balance de Clases (`Check_Class_Balance`)

## Descripción
Calcula la distribución de la variable objetivo `target_alta_venta` por zona geográfica.

## Objetivo
Evaluar el desbalance de clases antes del entrenamiento del modelo ML, permitiendo tomar decisiones sobre técnicas de balanceo (oversampling, undersampling, pesos de clase).

## Entrada
- Tabla: `GOLD_TABLE`

## Procesamiento
- Agrupación por `zona` y `target_alta_venta`
- Conteo de registros por clase
- Ordenamiento por zona y clase

## Salida
- Distribución de clases por zona geográfica

## Capa
**Gold (Validación ML)**


In [0]:
# Balance de las clases por zona
display(
    spark.sql(f"""
    SELECT
        zona,
        target_alta_venta,
        COUNT(*) AS total_registros
    FROM {GOLD_TABLE}
    GROUP BY zona, target_alta_venta
    ORDER BY zona, target_alta_venta
    """)
)


# Preparación Dataset ML (`Prepare_ML_Dataset`)

## Descripción
Construye el dataset final para entrenamiento del modelo de clasificación.

## Objetivo
Definir variables predictoras (features) y variable objetivo (`target_alta_venta`) orientadas a predecir si el importe de venta del siguiente periodo superará al actual.

## Entrada
- Tabla: `GOLD_TABLE`

## Procesamiento
- Selección de columnas:
  - Target: `target_alta_venta`
  - Features: variables numéricas, temporales y de comportamiento comercial
- Ordenamiento por `zona`, `fecha_pedido`

## Salida
- DataFrame: `df_ml`

## Capa
**Gold (ML)**


In [0]:
df_ml = spark.table(GOLD_TABLE)

feature_cols = [
    "ANIO",
    "MES",
    "DIA",
    "Unidades",
    "precio_unitario",
    "importe_coste_total",
    "lag_venta_1",
    "lag_venta_2",
    "lag_venta_3",
    "var_venta_1m",
    "var_venta_3m",
    "ma_3",
    "ma_7",
    "stddev_ventas_7m",
    "var_unidades_1m",
    "var_coste_1m",
    "venta_vs_ma_7",
    "margen_ganancia",
    "dias_entrega"
]

df_ml = df_ml.select("zona", "tipo_producto", "fecha_pedido", "target_alta_venta", *feature_cols)
display(df_ml.orderBy("zona", "fecha_pedido").limit(5))


# Definición Rango Temporal (`Define_Date_Range`)

## Descripción
Obtiene las fechas mínima y máxima del dataset de ventas.

## Objetivo
Establecer los límites temporales para la división cronológica train/test, garantizando la integridad de la serie de tiempo de ventas.

## Entrada
- DataFrame: `df_ml`

## Procesamiento
- Cálculo de `min(fecha_pedido)` y `max(fecha_pedido)`
- Extracción de valores a variables locales

## Salida
- `min_date`, `max_date`

## Capa
**Gold (Preparación ML)**


In [0]:
date_bounds = df_ml.agg(
    F.min("fecha_pedido").alias("min_date"),
    F.max("fecha_pedido").alias("max_date")
).collect()[0]

min_date = date_bounds["min_date"]
max_date = date_bounds["max_date"]

print("Fecha mínima:", min_date)
print("Fecha máxima:", max_date)


# División Cronológica Train/Test (`Time_Series_Split`)

## Descripción
Divide el dataset en conjuntos de entrenamiento y prueba respetando el orden cronológico de las ventas.

## Objetivo
Mantener la integridad temporal de la serie de ventas para evitar fuga de información (data leakage) en el entrenamiento del modelo.

## Entrada
- DataFrame: `df_ml`

## Procesamiento
- Obtención de fechas únicas ordenadas de `fecha_pedido`
- Cálculo del corte en el percentil 80% (80% train / 20% test)
- Definición de `split_date`

## Salida
- Fecha de corte: `split_date`

## Capa
**Gold (ML - Split Temporal)**


In [0]:
# Percentil temporal aproximado usando orden cronológico de fecha_pedido
distinct_dates = (
    df_ml.select("fecha_pedido")
    .distinct()
    .orderBy("fecha_pedido")
    .toPandas()["fecha_pedido"]
    .tolist()
)

split_idx  = int(len(distinct_dates) * 0.8)
split_date = distinct_dates[split_idx]

print("Fecha de corte (split_date):", split_date)


# Generación de Sets Train/Test (`Create_Train_Test_Sets`)

## Descripción
Separa el dataset en conjuntos de entrenamiento y prueba basándose en la fecha de corte cronológica.

## Objetivo
Preparar los datos para entrenamiento y evaluación del modelo de predicción de ventas, respetando el orden temporal.

## Entrada
- DataFrame: `df_ml`
- Fecha de corte: `split_date`

## Procesamiento
- `train_df`: registros con `fecha_pedido` anterior al split (80%)
- `test_df`: registros con `fecha_pedido` posterior o igual al split (20%)
- Conteo de registros por conjunto

## Salida
- DataFrames: `train_df`, `test_df`

## Capa
**Gold (ML)**


In [0]:
train_df = df_ml.filter(F.col("fecha_pedido") < F.lit(split_date))
test_df  = df_ml.filter(F.col("fecha_pedido") >= F.lit(split_date))

print("Registros de entrenamiento:", train_df.count())
print("Registros de prueba:", test_df.count())


# Pipeline de Modelado (`Build_ML_Pipeline`)

## Descripción
Construye un pipeline de ML con indexación de variables categóricas, ensamblado de features y Regresión Logística para clasificación de ventas.

## Objetivo
Predecir si el importe de venta del siguiente periodo superará al actual (`target_alta_venta = 1`), usando variables históricas de ventas, costos, márgenes y comportamiento temporal.

## Entrada
- DataFrames: `train_df`, `test_df`
- Features: `feature_cols`

## Procesamiento
- Codificación de variable categórica (`zona`)
- Ensamble de todas las variables en vector (`features`)
- Aplicación de modelo:
  - Regresión Logística (`maxIter=100`)

## Salida
- Pipeline: `lr_pipeline`

## Capa
**Gold (ML - Modelado)**


In [0]:
zona_indexer = StringIndexer(
    inputCol="zona",
    outputCol="zona_index",
    handleInvalid="keep"
)

assembler = VectorAssembler(
    inputCols=["zona_index"] + feature_cols,
    outputCol="features",
    handleInvalid="skip"
)

lr = LogisticRegression(
    featuresCol="features",
    labelCol="target_alta_venta",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=100,
    regParam=0.0
)

lr_pipeline = Pipeline(stages=[zona_indexer, assembler, lr])


# Entrenamiento y Predicción (`Train_Predict_Model`)

## Descripción
Entrena el modelo de Regresión Logística y genera predicciones sobre el conjunto de prueba.

## Objetivo
Evaluar la capacidad del modelo para predecir si el importe de venta del siguiente periodo será superior al actual.

## Entrada
- `train_df`, `test_df`
- Pipeline: `lr_pipeline`

## Procesamiento
- Entrenamiento del modelo (`fit`) con datos históricos de ventas
- Generación de predicciones (`transform`) sobre el conjunto de prueba

## Salida
- DataFrame: `lr_predictions` (incluye `prediction` y `probability`)

## Capa
**Gold (ML - Predicción)**


In [0]:
lr_model       = lr_pipeline.fit(train_df)
lr_predictions = lr_model.transform(test_df)

display(
    lr_predictions.select(
        "zona",
        "fecha_pedido",
        "target_alta_venta",
        "prediction",
        "probability"
    ).orderBy("zona", "fecha_pedido").limit(5)
)


# Entrenamiento y Predicción — Verificación (`Train_Predict_Model`)

## Descripción
Reejecutar el entrenamiento y predicción del modelo de Regresión Logística para confirmación de resultados.

## Objetivo
Predecir si el importe de venta del siguiente periodo supera al actual (`target_alta_venta`).

## Entrada
- `train_df`, `test_df`
- Pipeline: `lr_pipeline`

## Procesamiento
- Entrenamiento del modelo (`fit`)
- Generación de predicciones (`transform`)

## Salida
- DataFrame: `lr_predictions` (incluye `prediction`, `probability`)

## Capa
**Gold (ML)**


In [0]:
lr_model       = lr_pipeline.fit(train_df)
lr_predictions = lr_model.transform(test_df)

display(
    lr_predictions.select(
        "zona",
        "fecha_pedido",
        "target_alta_venta",
        "prediction",
        "probability"
    ).orderBy("zona", "fecha_pedido").limit(5)
)


# Evaluación del Modelo (`Evaluate_Model`)

## Descripción
Define los evaluadores de métricas para medir el desempeño del modelo de clasificación de ventas.

## Objetivo
Cuantificar la capacidad predictiva del modelo sobre datos históricos de ventas.

## Entrada
- DataFrame: `lr_predictions`

## Procesamiento
- Métricas calculadas:
  - AUC (ROC): discriminación entre alta y baja venta
  - Accuracy: porcentaje de predicciones correctas
  - F1 Score: balance entre precision y recall
  - Precision ponderada
  - Recall ponderado

## Salida
- Evaluadores y métricas (`auc_lr`, evaluadores multi-clase)

## Capa
**Gold (ML - Evaluación)**


In [0]:
binary_eval = BinaryClassificationEvaluator(
    labelCol="target_alta_venta",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc_lr = binary_eval.evaluate(lr_predictions)

accuracy_eval = MulticlassClassificationEvaluator(
    labelCol="target_alta_venta",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_eval = MulticlassClassificationEvaluator(
    labelCol="target_alta_venta",
    predictionCol="prediction",
    metricName="f1"
)

precision_eval = MulticlassClassificationEvaluator(
    labelCol="target_alta_venta",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall_eval = MulticlassClassificationEvaluator(
    labelCol="target_alta_venta",
    predictionCol="prediction",
    metricName="weightedRecall"
)


# Resultados del Modelo (`Model_Metrics_Results`)

## Descripción
Calcula y muestra las métricas finales del modelo de Regresión Logística sobre el conjunto de prueba de ventas.

## Objetivo
Cuantificar el desempeño del modelo de clasificación de alta/baja venta.

## Entrada
- DataFrame: `lr_predictions`

## Procesamiento
- Evaluación de métricas:
  - AUC ROC
  - Accuracy
  - F1-score
  - Precision ponderada
  - Recall ponderado

## Salida
- Métricas impresas en consola

## Capa
**Gold (ML - Evaluación)**


In [0]:
accuracy_lr  = accuracy_eval.evaluate(lr_predictions)
f1_lr        = f1_eval.evaluate(lr_predictions)
precision_lr = precision_eval.evaluate(lr_predictions)
recall_lr    = recall_eval.evaluate(lr_predictions)

print(f"AUC ROC (Logistic Regression): {auc_lr:.4f}")
print(f"Accuracy (Logistic Regression): {accuracy_lr:.4f}")
print(f"F1-score (Logistic Regression): {f1_lr:.4f}")
print(f"Weighted Precision (Logistic Regression): {precision_lr:.4f}")
print(f"Weighted Recall (Logistic Regression): {recall_lr:.4f}")


# Modelo Random Forest (`Build_RF_Model`)

## Descripción
Implementa un modelo de clasificación basado en Random Forest para predecir el comportamiento de ventas.

## Objetivo
Capturar relaciones no lineales entre las variables de ventas, costos, márgenes y comportamiento temporal para mejorar la capacidad predictiva frente a la Regresión Logística.

## Entrada
- `train_df`, `test_df`
- Features: `feature_cols`

## Procesamiento
- Ensamble de variables en vector `features`
- Entrenamiento Random Forest (`numTrees=100`, `maxDepth=6`, `seed=42`)
- Generación de predicciones sobre el conjunto de prueba

## Salida
- DataFrame: `rf_predictions`

## Capa
**Gold (ML - Modelado)**


In [0]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="target_alta_venta",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    numTrees=100,
    maxDepth=6,
    seed=42
)

rf_pipeline = Pipeline(stages=[zona_indexer, assembler, rf])
rf_model    = rf_pipeline.fit(train_df)
rf_predictions = rf_model.transform(test_df)


# Evaluación Random Forest (`Evaluate_RF_Model`)

## Descripción
Calcula las métricas de desempeño del modelo Random Forest sobre el conjunto de prueba de ventas.

## Objetivo
Comparar el rendimiento del Random Forest frente a la Regresión Logística en la predicción de alta/baja venta.

## Entrada
- DataFrame: `rf_predictions`

## Procesamiento
- Evaluación de métricas:
  - AUC ROC
  - Accuracy
  - F1-score
  - Precision ponderada
  - Recall ponderado

## Salida
- Métricas impresas en consola

## Capa
**Gold (ML - Evaluación)**


In [0]:
auc_rf       = binary_eval.evaluate(rf_predictions)
accuracy_rf  = accuracy_eval.evaluate(rf_predictions)
f1_rf        = f1_eval.evaluate(rf_predictions)
precision_rf = precision_eval.evaluate(rf_predictions)
recall_rf    = recall_eval.evaluate(rf_predictions)

print(f"AUC ROC (Random Forest): {auc_rf:.4f}")
print(f"Accuracy (Random Forest): {accuracy_rf:.4f}")
print(f"F1-score (Random Forest): {f1_rf:.4f}")
print(f"Weighted Precision (Random Forest): {precision_rf:.4f}")
print(f"Weighted Recall (Random Forest): {recall_rf:.4f}")


# Comparación de Modelos (`Compare_Models`)

## Descripción
Consolida y compara las métricas de los modelos evaluados sobre el dataset de ventas.

## Objetivo
Identificar el modelo con mejor desempeño predictivo para la clasificación de alta/baja venta en el contexto comercial.

## Entrada
- Métricas: Logistic Regression, Random Forest

## Procesamiento
- Creación de DataFrame comparativo con métricas por modelo
- Ordenamiento y visualización

## Salida
- DataFrame: `metrics_df`

## Capa
**Gold (ML - Evaluación Comparativa)**


A continuacion se muestra el desempeño los resultados del modelo de clasificación para predecir el Target de Alta Venta. Los resultados indican que se cuenta con un modelo con una capacidad predictiva sólida, pero con áreas de mejora en la precisión de las clases positivas.

## Evaluación del Modelo de Machine Learning: Resultados de Clasificación
Explicación del Procedimiento:
Tras entrenar el modelo en la capa Gold utilizando un split temporal, se procedió a evaluar su capacidad de generalización sobre el conjunto de prueba (test). Se generó una Matriz de Confusión para comparar las predicciones del modelo frente a los valores reales y un Reporte de Clasificación que detalla métricas clave como Precision, Recall y F1-Score para ambas clases (0: Venta Normal, 1: Alta Venta).

Análisis de la Matriz de Confusión (Outputs):
Verdaderos Negativos (1177): El modelo es excelente identificando las ventas que no serán excepcionalmente altas.

Verdaderos Positivos (719): El modelo logra capturar una cantidad importante de casos de éxito comercial.

Falsos Positivos (218): En 218 casos, el modelo predijo una "Alta Venta" que no ocurrió. Esto podría llevar a un exceso de inventario o sobreestimación de ingresos.

Falsos Negativos (304): Estos son "costos de oportunidad". El modelo no detectó 304 ventas altas, lo que significa que la empresa no se preparó para esa demanda extra.

Interpretación de Métricas:
Accuracy (79%): El modelo acierta en casi 8 de cada 10 predicciones. Para un modelo de Big Data con alta variabilidad, es un resultado inicial muy prometedor.

Precision - Clase 1 (77%): Cuando el modelo dice que habrá una "Alta Venta", tiene un 77% de probabilidad de estar en lo cierto.

Recall - Clase 1 (70%): El modelo es capaz de detectar el 70% del total de las altas ventas reales. Este es el punto principal a mejorar; idealmente, querríamos capturar más casos positivos.

F1-Score (0.73): El balance entre precisión y sensibilidad es saludable, lo que indica que el modelo no está sesgado excesivamente hacia ninguna de las dos clases.

Conclusiones y Siguientes Pasos:
Reflexión Técnica: La diferencia entre el acierto de la clase 0 (80%) y la clase 1 (70%) sugiere que los patrones de "Venta Normal" son más estables y fáciles de aprender que los de "Alta Venta", los cuales suelen estar influenciados por factores externos no presentes en el dataset (promociones, factores políticos, etc.).

In [0]:
metrics_data = [
    ("LogisticRegression", auc_lr, accuracy_lr, f1_lr, precision_lr, recall_lr),
    ("RandomForest",       auc_rf, accuracy_rf, f1_rf, precision_rf, recall_rf),
]

metrics_df = spark.createDataFrame(
    metrics_data,
    ["modelo", "auc_roc", "accuracy", "f1_score", "precision_ponderada", "recall_ponderado"]
)

display(metrics_df)


# Reflexión General
El modelo Random Forest aplicado al dataset de ventas históricas globales busca capturar patrones comerciales no lineales relacionados con zonas geográficas, tipos de producto, canales de venta, márgenes y comportamiento temporal. La calidad predictiva dependerá de la riqueza de las features generadas y de la consistencia del dato histórico procesado en las capas Bronze y Silver.

# Conclusiones del Modelo Random Forest

## 1. Capacidad discriminativa
Un **AUC ROC cercano o superior a 0.6** indica que el modelo comienza a capturar señales comerciales relevantes. Valores por debajo de 0.5 sugieren que las features seleccionadas no son suficientemente informativas para el horizonte de predicción definido.

## 2. Precisión global
**Accuracy esperado: entre 50% y 70%** dependiendo del balanceo de clases y la riqueza de las variables predictoras. Un accuracy inferior al 50% indica que el modelo no supera un clasificador trivial.

## 3. Desempeño por métricas
- **F1-score:** Mide el balance entre precisión y recall. Valores bajos pueden indicar desbalance de clases o features poco informativas.
- **Precision:** Calidad de las predicciones positivas (ventas altas identificadas correctamente).
- **Recall:** Capacidad del modelo para detectar todos los periodos de alta venta real.

## 4. Posibles causas de bajo desempeño
Desde una perspectiva de Data Science aplicada a ventas:
- Granularidad del dataset (nivel pedido vs. nivel agregado mensual)
- Features poco informativas o con alta correlación entre sí
- Desbalance de clases en `target_alta_venta`
- Horizonte de predicción a corto plazo con alta variabilidad
- Estacionalidad no capturada adecuadamente en los lags

## 5. Implicaciones y recomendaciones
- Enriquecer features con variables de estacionalidad (trimestre, festivos, campañas)
- Agregar datos externos (macro-económicos, tasas de cambio por región)
- Explorar modelos de series de tiempo (Prophet, LSTM) para capturar patrones temporales complejos
- Evaluar la predicción a nivel mensual o trimestral en lugar de por pedido
- Aplicar técnicas de balanceo de clases (SMOTE, pesos de clase) si hay desbalance significativo
